In [0]:
from pyspark.sql import functions as F
import re

BUCKET = "football-analytics-nirmal"   # Your bucket name (must be in an external location)
PREFIX = "bronze/premier-league/2024-25"

S3 = {
    "standings": f"s3a://{BUCKET}/{PREFIX}/standings/standings.csv",
    "matches":   f"s3a://{BUCKET}/{PREFIX}/matches/matches.csv",
    "players":   f"s3a://{BUCKET}/{PREFIX}/players/players.csv",
}

def clean_colname(col):
    """Clean a column name: replace invalid chars with underscores (Delta/UC safe)."""
    # Only allow alphanumeric and underscores
    return re.sub(r'[^A-Za-z0-9_]', '_', col.strip().replace(' ', '_'))

def clean_column_names(df):
    """Return a copy of the DataFrame with cleaned column names."""
    new_names = [clean_colname(c) for c in df.columns]
    for old, new in zip(df.columns, new_names):
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df

def read_bronze(path, table_name):
    print(f"\n  Reading: {path.split('/')[-1]}")
    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("multiLine", True)
            .option("escape", '"')
            .option("encoding", "UTF-8")
            .csv(path)
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.lit(path))
            .withColumn("_layer", F.lit("bronze"))
    )
    df = clean_column_names(df)
    row_count = df.count()
    (
        df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"football.pl_2024_25.{table_name}")
    )
    print(f"  ✅ football.pl_2024_25.{table_name:25s} → {row_count:>5,} rows")
    print(f"     Columns: {df.columns}")
    return df

print("=" * 55)
print("  BRONZE LAYER — Reading from S3 via Unity Catalog (clean colnames)")
print("=" * 55)

b_standings = read_bronze(S3["standings"], "bronze_standings")
b_matches   = read_bronze(S3["matches"],   "bronze_matches")
b_players   = read_bronze(S3["players"],   "bronze_players")

print("\n✅ Bronze layer complete")


  BRONZE LAYER — Reading from S3 via Unity Catalog (clean colnames)

  Reading: standings.csv
  ✅ football.pl_2024_25.bronze_standings          →    20 rows
     Columns: ['Rk', 'Squad', 'MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Pts_MP', 'xG', 'xGA', 'xGD', 'xGD_90', 'Attendance', 'Top_Team_Scorer', 'Goalkeeper', 'Notes', '_ingested_at', '_source_file', '_layer']

  Reading: matches.csv
  ✅ football.pl_2024_25.bronze_matches            → 9,380 rows
     Columns: ['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals', 'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals', 'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots', 'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners', 'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards', 'HomeRedCards', 'AwayRedCards', '_ingested_at', '_source_file', '_layer']

  Reading: players.csv
  ✅ football.pl_2024_25.bronze_players            → 4,270 rows
     Columns: ['Player', 'Team', '_',

In [0]:

print("── STANDINGS columns ──")
spark.read.table("football.pl_2024_25.bronze_standings").printSchema()

print("\n── MATCHES columns ──")
spark.read.table("football.pl_2024_25.bronze_matches").printSchema()

print("\n── PLAYERS columns ──")
spark.read.table("football.pl_2024_25.bronze_players").printSchema()

── STANDINGS columns ──
root
 |-- Rk: integer (nullable = true)
 |-- Squad: string (nullable = true)
 |-- MP: integer (nullable = true)
 |-- W: integer (nullable = true)
 |-- D: integer (nullable = true)
 |-- L: integer (nullable = true)
 |-- GF: integer (nullable = true)
 |-- GA: integer (nullable = true)
 |-- GD: integer (nullable = true)
 |-- Pts: integer (nullable = true)
 |-- Pts_MP: double (nullable = true)
 |-- xG: double (nullable = true)
 |-- xGA: double (nullable = true)
 |-- xGD: double (nullable = true)
 |-- xGD_90: double (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- Top_Team_Scorer: string (nullable = true)
 |-- Goalkeeper: string (nullable = true)
 |-- Notes: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _layer: string (nullable = true)


── MATCHES columns ──
root
 |-- Season: string (nullable = true)
 |-- MatchDate: date (nullable = true)
 |-- HomeTeam: string (nullable = tru

In [0]:
bronze_standings = spark.table("football.pl_2024_25.bronze_standings")

silver_standings = (
    bronze_standings
    .filter(F.col("Squad").isNotNull())

    # Rename columns
    .withColumnRenamed("Squad", "team")
    # Cast numeric fields
    .withColumn("rank", F.col("Rk").cast("int"))
    .withColumn("matches_played", F.col("MP").cast("int"))
    .withColumn("wins", F.col("W").cast("int"))
    .withColumn("draws", F.col("D").cast("int"))
    .withColumn("losses", F.col("L").cast("int"))
    .withColumn("goals_for", F.col("GF").cast("int"))
    .withColumn("goals_against", F.col("GA").cast("int"))
    .withColumn("goal_diff", F.col("GD").cast("int"))
    .withColumn("points", F.col("Pts").cast("int"))
    .withColumn("expected_goals", F.col("xG").cast("double"))
    .withColumn("expected_goals_against", F.col("xGA").cast("double"))
    # Derived metrics
    .withColumn("goals_vs_xg", F.round(F.col("goals_for") - F.col("expected_goals"), 2))
    .withColumn("goals_conceded_vs_xga", F.round(F.col("goals_against") - F.col("expected_goals_against"), 2))
    # Split 'Top_Team_Scorer' from rightmost '-' into 'scorer' and 'goals'
    # Split 'Top_Team_Scorer' from rightmost '-' into 'scorer' and 'goals'
    .withColumn(
        "scorer",
        F.trim(
            F.expr(
                "CASE WHEN Top_Team_Scorer IS NOT NULL THEN "
                "  SPLIT(Top_Team_Scorer, '-', -1)[0] "
                "ELSE NULL END"
            )
        )
    )
    .withColumn(
        "goals",
        F.expr(
            "CASE WHEN Top_Team_Scorer IS NOT NULL THEN "
            "  CAST(trim(SPLIT(Top_Team_Scorer, '-', -1)[size(SPLIT(Top_Team_Scorer, '-', -1))-1]) AS int) "
            "ELSE NULL END"
        )
    )
    # League zones
    .withColumn(
        "zone",
        F.when(F.col("rank") <= 4, "Champions League")
         .when(F.col("rank") <= 6, "Europa League")
         .when(F.col("rank") >= 18, "Relegation")
         .otherwise("Mid-Table")
    )
)

# -------------------------------
# SILVER: MATCHES
# -------------------------------
bronze_matches = spark.table("football.pl_2024_25.bronze_matches")

silver_matches = (
    bronze_matches
    .filter(F.col("HomeTeam").isNotNull())
    .filter(F.col("AwayTeam").isNotNull())
    .withColumn("home_goals", F.col("FullTimeHomeGoals").cast("int"))
    .withColumn("away_goals", F.col("FullTimeAwayGoals").cast("int"))
    .withColumn("total_goals", F.col("home_goals") + F.col("away_goals"))
    .withColumn(
        "result",
        F.when(F.col("home_goals") > F.col("away_goals"), "Home Win")
         .when(F.col("home_goals") < F.col("away_goals"), "Away Win")
         .otherwise("Draw")
    )
    .withColumn("match_date", F.col("MatchDate"))
)

# -------------------------------
# SILVER: PLAYERS
# -------------------------------
bronze_players = spark.table("football.pl_2024_25.bronze_players")

silver_players = (
    bronze_players
    .filter(F.col("Player").isNotNull())
    .filter(F.col("Team").isNotNull())
    .withColumn(
        "age",
        F.when(
            F.col("Age").isNotNull() & F.col("Age").rlike(r"^\d+-\d+$"),
            F.round(
                F.split(F.col("Age"), "-")[0].cast("int") +
                (F.split(F.col("Age"), "-")[1].cast("int") / 365),
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn("minutes", F.coalesce(F.col("Minutes").cast("int"), F.lit(0.0)))
    .withColumnRenamed("_", "Jersey_Number")
    .withColumn("Jersey_Number", F.col("Jersey_Number").cast("int"))
    .withColumn("goals", F.coalesce(F.col("Goals").cast("int"), F.lit(0.0)))
    .withColumn("assists", F.coalesce(F.col("Assists").cast("int"), F.lit(0.0)))
)

# -------------------------------
# WRITE SILVER TABLES
# -------------------------------
silver_standings.write.mode("overwrite").saveAsTable("football.pl_2024_25.silver_standings")
silver_matches.write.mode("overwrite").saveAsTable("football.pl_2024_25.silver_matches")
silver_players.write.mode("overwrite").saveAsTable("football.pl_2024_25.silver_players")

print("✅ Silver layer created successfully")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6677445697874808>, line 107
    105 silver_standings.write.mode("overwrite").saveAsTable("football.pl_2024_25.silver_standings")
    106 silver_matches.write.mode("overwrite").saveAsTable("football.pl_2024_25.silver_matches")
--> 107 silver_players.write.mode("overwrite").saveAsTable("football.pl_2024_25.silver_players")
    109 print("✅ Silver layer created successfully")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/py

# Removed all references to 'role', replaced with 'Position'. Column names match actual schema based on verified sample.

In [0]:
print("📦 All tables in football.pl_2024_25:")
spark.sql("SHOW TABLES IN football.pl_2024_25").show(truncate=False)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🔧 Fix Silver & Gold Player Tables

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Step 1: Find the real minutes column name ─────────────────────────────────
bronze = spark.read.table("football.pl_2024_25.bronze_players")

print("All column names in bronze_players:")
for c in bronze.columns:
    print(f"  '{c}'")

# COMMAND ----------

# ── Step 2: Show sample to confirm correct columns ────────────────────────────
bronze.show(3, truncate=False)

# COMMAND ----------

# ── Step 3: Rebuild Silver with correct column mapping ────────────────────────
bronze = spark.read.table("football.pl_2024_25.bronze_players")

mins_col = next(
    (c for c in bronze.columns if c.lower() in
     ["mins", "min", "minutes", "mins_played", "minutesplayed",
      "minutes_played", "time", "mins.", "90s"]),
    None
)

print(f"✅ Found minutes column: '{mins_col}'")
if mins_col is None:
    print("⚠️  No minutes column found — printing all columns:")
    print(bronze.columns)
    raise Exception("Cannot find minutes column — check column names above and update mins_col manually")

# Fix role using FIFA position codes
def map_position_to_role(pos_col):
    return (
        F.when(pos_col.isin("GK"), "Goalkeeper")
         .when(pos_col.rlike("(?i)^(CB|RB|LB|WB|SW|RWB|LWB|DC|DR|DL)"), "Defender")
         .when(pos_col.rlike("(?i)^(CM|DM|AM|LM|RM|CDM|CAM|LAM|RAM|MC|ML|MR)"), "Midfielder")
         .when(pos_col.rlike("(?i)^(ST|CF|RW|LW|FW|SS|RF|LF|AML|AMR|AMC)"), "Forward")
         .when(pos_col.rlike("(?i)(FW|ST|CF|RW|LW)"), "Forward")
         .when(pos_col.rlike("(?i)(MF|CM|DM|AM|LM|RM)"), "Midfielder")
         .when(pos_col.rlike("(?i)(DF|CB|RB|LB|WB)"), "Defender")
         .otherwise("Unknown")
    )

silver_players_fixed = (
    bronze
    .filter(F.col("Player").isNotNull())
    .filter(F.col("Team").isNotNull())
    .withColumnRenamed("_", "Jersey_Number")
    .withColumn("Jersey_Number", F.col("Jersey_Number").cast("int"))

    # Fix minutes — strip commas, cast to double
    .withColumn("minutes",
        F.regexp_replace(F.col(mins_col).cast("string"), ",", "").cast("double"))

    .withColumn("goals",
        F.coalesce(F.col("Goals").cast("double"), F.lit(0.0)))
    .withColumn("assists",
        F.coalesce(F.col("Assists").cast("double"), F.lit(0.0)))
    .withColumn("expected_goals",
        F.coalesce(F.col("Expected_Goals__xG_").cast("double"), F.lit(0.0)))
    .withColumn("expected_assists",
        F.coalesce(F.col("Expected_Assists__xAG_").cast("double"), F.lit(0.0)))
    .withColumn("matches_played",
        F.lit(None).cast("int"))  # Not present, add placeholder
    .withColumn("yellow_cards",
        F.coalesce(F.col("Yellow_Cards").cast("int"), F.lit(0)))
    .withColumn("red_cards",
        F.coalesce(F.col("Red_Cards").cast("int"), F.lit(0)))
       .withColumn(
        "age",
        F.when(
            F.col("Age").isNotNull() & F.col("Age").rlike(r"^\d+-\d+$"),
            F.round(
                F.split(F.col("Age"), "-")[0].cast("int") +
                (F.split(F.col("Age"), "-")[1].cast("int") / 365),
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn("position",
        F.col("Position"))
    .withColumn("role", map_position_to_role(F.col("Position")))

    # Derived metrics
    .withColumn("goals_above_xg",
        F.when(F.col("Expected_Goals__xG_").isNotNull(),
               F.round(F.col("Goals") - F.col("Expected_Goals__xG_"), 2))
         .otherwise(F.lit(0.0)))
    .withColumn("contributions_per90",
        F.when(F.col("minutes") > 0,
               F.round((F.col("Goals") + F.col("Assists")) / (F.col("minutes") / 90), 2))
         .otherwise(F.lit(0.0)))

    .drop("_ingested_at", "_source_file", "_layer")
)

# Save fixed Silver
(silver_players_fixed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("football.pl_2024_25.silver_players")
)

total     = silver_players_fixed.count()
has_mins  = silver_players_fixed.filter(F.col("minutes") > 0).count()
role_dist = silver_players_fixed.groupBy("role").count().orderBy("count", ascending=False)

print(f"\n✅ silver_players fixed: {total} players")
print(f"   Players with minutes > 0: {has_mins}")
print("\n── Role distribution ──")
role_dist.show()

# COMMAND ----------

# ── Step 4: Rebuild Gold Player Rankings ─────────────────────────────────────
silver = spark.read.table("football.pl_2024_25.silver_players")

qualifying = silver.filter(F.col("minutes") >= 450).filter(F.col("role") != "Goalkeeper")
print(f"Players qualifying (>= 450 mins, non-GK): {qualifying.count()}")

if qualifying.count() < 10:
    print("\n⚠️  Low count — checking minutes distribution:")
    silver.selectExpr(
        "COUNT(*) as total",
        "SUM(CASE WHEN minutes >= 450 THEN 1 ELSE 0 END) as over_450",
        "SUM(CASE WHEN minutes >= 90  THEN 1 ELSE 0 END) as over_90",
        "MIN(minutes) as min_mins",
        "MAX(minutes) as max_mins",
        "AVG(minutes) as avg_mins"
    ).show()
    qualifying = silver.filter(F.col("minutes") >= 90).filter(F.col("role") != "Goalkeeper")
    print(f"Lowered to >= 90 mins: {qualifying.count()} players")

gold_players = (
    qualifying
    .withColumn("rank_goals",
        F.rank().over(Window.orderBy(F.desc("goals"))))
    .withColumn("rank_assists",
        F.rank().over(Window.orderBy(F.desc("assists"))))
    .withColumn("rank_contributions",
        F.rank().over(Window.orderBy(F.desc("contributions_per90"))))
    .withColumn("rank_goals_vs_xg",
        F.rank().over(Window.orderBy(F.desc("goals_above_xg"))))
    .withColumn("composite_score",
        F.round(
            (F.col("goals") * 3)
          + (F.col("assists") * 2)
          + (F.col("goals_above_xg") * 2)
          + (F.col("contributions_per90") * 5), 2))
    .withColumn("overall_rank",
        F.rank().over(Window.orderBy(F.desc("composite_score"))))
    .orderBy("overall_rank")
)

(gold_players.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("football.pl_2024_25.gold_player_rankings")
)

print(f"\n✅ gold_player_rankings rebuilt: {gold_players.count()} players")
print("\n── Top 10 players ──")
gold_players.select(
    "overall_rank", "Player", "Team", "role",
    "minutes", "goals", "assists", "contributions_per90"
).show(10, truncate=False)

All column names in bronze_players:
  'Player'
  'Team'
  '_'
  'Nation'
  'Position'
  'Age'
  'Minutes'
  'Goals'
  'Assists'
  'Penalty_Shoot_on_Goal'
  'Penalty_Shoot'
  'Total_Shoot'
  'Shoot_on_Target'
  'Yellow_Cards'
  'Red_Cards'
  'Touches'
  'Dribbles'
  'Tackles'
  'Blocks'
  'Expected_Goals__xG_'
  'Non_Penalty_xG__npxG_'
  'Expected_Assists__xAG_'
  'Shot_Creating_Actions'
  'Goal_Creating_Actions'
  'Passes_Completed'
  'Passes_Attempted'
  'Pass_Completion__'
  'Progressive_Passes'
  'Carries'
  'Progressive_Carries'
  'Dribble_Attempts'
  'Successful_Dribbles'
  'Date'
  '_ingested_at'
  '_source_file'
  '_layer'
+---------------+-----------------+---+------+--------+------+-------+-----+-------+---------------------+-------------+-----------+---------------+------------+---------+-------+--------+-------+------+-------------------+---------------------+----------------------+---------------------+---------------------+----------------+----------------+----------------

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



✅ gold_player_rankings rebuilt: 1601 players

── Top 10 players ──
+------------+--------------------+-----------------------+----------+-------+-----+-------+-------------------+
|overall_rank|Player              |Team                   |role      |minutes|goals|assists|contributions_per90|
+------------+--------------------+-----------------------+----------+-------+-----+-------+-------------------+
|1           |Cole Palmer         |Chelsea                |Midfielder|90.0   |4.0  |0.0    |4.0                |
|2           |Noni Madueke        |Chelsea                |Forward   |90.0   |3.0  |0.0    |3.0                |
|3           |Erling Haaland      |Manchester City        |Forward   |90.0   |3.0  |0.0    |3.0                |
|4           |Mohamed Salah       |Liverpool              |Forward   |90.0   |2.0  |1.0    |3.0                |
|5           |Bukayo Saka         |Arsenal                |Midfielder|90.0   |1.0  |2.0    |3.0                |
|6           |Mohamed Salah 

In [0]:
# ── GOLD: TEAM PERFORMANCE ────────────────────────────────────────────────────
home_stats = (
    spark.read.table("football.pl_2024_25.silver_matches")
    .groupBy(F.col("HomeTeam").alias("team"))
    .agg(
        F.count("*").alias("home_games"),
        F.sum(F.when(F.col("result") == "Home Win", 1).otherwise(0)).alias("home_wins"),
        F.sum(F.when(F.col("result") == "Draw",     1).otherwise(0)).alias("home_draws"),
        F.sum(F.when(F.col("result") == "Away Win", 1).otherwise(0)).alias("home_losses"),
        F.sum("home_goals").alias("home_goals_scored"),
        F.sum("away_goals").alias("home_goals_conceded"),
        F.round(F.avg("home_goals"), 2).alias("avg_home_goals"),
    )
)
 
away_stats = (
    spark.read.table("football.pl_2024_25.silver_matches")
    .groupBy(F.col("AwayTeam").alias("team"))
    .agg(
        F.count("*").alias("away_games"),
        F.sum(F.when(F.col("result") == "Away Win", 1).otherwise(0)).alias("away_wins"),
        F.sum(F.when(F.col("result") == "Draw",     1).otherwise(0)).alias("away_draws"),
        F.sum(F.when(F.col("result") == "Home Win", 1).otherwise(0)).alias("away_losses"),
        F.sum("away_goals").alias("away_goals_scored"),
        F.sum("home_goals").alias("away_goals_conceded"),
        F.round(F.avg("away_goals"), 2).alias("avg_away_goals"),
    )
)
 
gold_teams = (
    spark.read.table("football.pl_2024_25.silver_standings")
    .join(home_stats, on="team", how="left")
    .join(away_stats, on="team", how="left")
    .withColumn("home_win_rate",
        F.round(F.col("home_wins") / F.col("home_games") * 100, 1))
    .withColumn("away_win_rate",
        F.round(F.col("away_wins") / F.col("away_games") * 100, 1))
    .withColumn("overall_win_rate",
        F.round(F.col("wins") / F.col("matches_played") * 100, 1))
    .withColumn("home_away_gap",
        F.round(F.col("home_win_rate") - F.col("away_win_rate"), 1))
    .withColumn("home_away_profile",
        F.when(F.col("home_away_gap") >  25, "🏟️ Fortress at home")
         .when(F.col("home_away_gap") < -10, "✈️ Better away")
         .otherwise("⚖️ Balanced"))
    .orderBy("rank")
)
 
gold_teams.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("football.pl_2024_25.gold_team_performance")

In [0]:
  %sql
  WITH avg_stats AS (
      SELECT
          AVG(goals_for)     AS avg_gf,
          AVG(goals_against) AS avg_ga
      FROM football.pl_2024_25.silver_standings
  )
  SELECT
      s.rank,
      s.team,
      s.goals_for,
      s.goals_against,
      s.points,
      ROUND(s.goals_for     / s.matches_played, 2) AS scored_per_game,
      ROUND(s.goals_against / s.matches_played, 2) AS conceded_per_game,
      CASE
          WHEN s.goals_for > a.avg_gf AND s.goals_against < a.avg_ga THEN '⚽🛡️ Complete'
          WHEN s.goals_for > a.avg_gf AND s.goals_against > a.avg_ga THEN '⚽💥 Attack-minded'
          WHEN s.goals_for < a.avg_gf AND s.goals_against < a.avg_ga THEN '🛡️ Defensive'
          ELSE '📉 Struggling'
      END AS team_profile
  FROM football.pl_2024_25.silver_standings s
  CROSS JOIN avg_stats a
  ORDER BY s.rank;

rank,team,goals_for,goals_against,points,scored_per_game,conceded_per_game,team_profile
1,Liverpool,86,41,84,2.26,1.08,⚽🛡️ Complete
2,Arsenal,69,34,74,1.82,0.89,⚽🛡️ Complete
3,Manchester City,72,44,71,1.89,1.16,⚽🛡️ Complete
4,Chelsea,64,43,69,1.68,1.13,⚽🛡️ Complete
5,Newcastle Utd,68,47,66,1.79,1.24,⚽🛡️ Complete
6,Aston Villa,58,51,66,1.53,1.34,⚽🛡️ Complete
7,Nott'ham Forest,58,46,65,1.53,1.21,⚽🛡️ Complete
8,Brighton,66,59,61,1.74,1.55,⚽💥 Attack-minded
9,Bournemouth,58,46,56,1.53,1.21,⚽🛡️ Complete
10,Brentford,66,57,56,1.74,1.5,⚽💥 Attack-minded
